# Fair Job Recommender System (FJRS)
### Exposure Inequality Study - Notebook

---

## Project Overview

This notebook is part of a study on **algorithmic fairness in job recommendation systems**. The goal is to analyse whether a standard collaborative filtering model treats different groups of job seekers fairly. In particular, we investigate whether some groups receive less exposure to **high-opportunity jobs** in ranked recommendation lists.

---

## What this notebook covers

| Step | Description |
|---|---|
| **Step 0** | Setup and imports |
| **Step 1** | Load cleaned data |
| **Step 2** | Filter active users and popular jobs, remap IDs |
| **Step 3** | Leave-last-out train/test split |
| **Step 4** | Baseline model: Implicit Matrix Factorization (BPR-SGD) |
| **Step 5** | Train the baseline model |
| **Step 6** | Generate baseline recommendations |
| **Step 7** | Accuracy metrics (Precision@K, Recall@K, nDCG@K) |
| **Step 8** | Exposure fairness metrics (position-based exposure parity) |
| **Step 9** | Fairness-aware re-ranking (greedy approach with tunable beta) |
| **Step 10** | Evaluate the fairness-aware model |
| **Step 11** | Compare baseline and fairness-aware models |
| **Step 12** | Trade-off analysis by sweeping beta |
| **Step 13** | Conclusion |

---

## Key concepts

**Implicit feedback** - Users did not explicitly rate jobs. Instead, when a user applies to a job this is treated as a positive signal (rating = 1). Jobs that were not applied to are considered unknown rather than negative.

**Groups** - Derived from the `ManagedOthers` field in the user profile:
- **Group A**: users who have managed others (proxy for more experienced applicants)
- **Group B**: users who have not managed others

**Job tier** - Created using keywords in job titles:
- **high_opportunity**: roles such as senior, lead, manager, engineer, analyst, etc.
- **standard**: all other roles

**Exposure** - Items that appear higher in a recommendation list receive more attention. We model this with a position discount: `exposure(rank) = 1 / log2(rank + 2)`. Fairness is evaluated by measuring the difference in average exposure to high-opportunity jobs between Group A and Group B.

---

## Step 0 - Setup & Imports

Import all required libraries. The notebook uses pandas for data handling, numpy for numerical operations, matplotlib for plotting, and tqdm for progress bars.

In [33]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

---

## Step 1 - Load Cleaned Data

The cleaned dataset is loaded from the `data/` directory, which contains the prepared parquet files.

### Input
Files in `data/`:

| File | Description |
|---|---|
| `interactions.parquet` | One row per (user, job) application. Columns: `user_id`, `job_id`, `rating` |
| `users.parquet` | One row per user. Columns: `user_id`, `group` (`A` or `B`) |
| `jobs.parquet` | One row per job. Columns: `job_id`, `tier` (`high_opportunity` or `standard`) |
| `user_history_agg.parquet` | Aggregated past job titles per user (for future content-based extensions) |
| `meta.json` | Scalars `N_USERS` and `N_JOBS` |

In [36]:
data_dir = os.path.join('..', 'data')

interactions = pd.read_parquet(os.path.join(data_dir, 'interactions.parquet'))
users = pd.read_parquet(os.path.join(data_dir, 'users.parquet'))
jobs = pd.read_parquet(os.path.join(data_dir, 'jobs.parquet'))

print(f' interactions : {len(interactions):,} rows')
print(f' users        : {len(users):,} rows')
print(f' jobs         : {len(jobs):,} rows')
print()
print('Group distribution:')
print(users['group'].value_counts().to_string())
print()
print('Tier distribution:')
print(jobs['tier'].value_counts().to_string())

 interactions : 1,417,471 rows
 users        : 308,022 rows
 jobs         : 349,712 rows

Group distribution:
group
B    228200
A     79822

Tier distribution:
tier
standard            219388
high_opportunity    130324


---

## Step 2 - Filter Active Users & Popular Jobs, Remap IDs

The raw dataset is very sparse: most users applied to only a handful of jobs and most jobs received very few applications. Training a latent-factor model on such a large, sparse matrix leads to poor-quality embeddings and near-zero evaluation metrics.

### Filtering strategy
We apply a **k-core filter**: keep only users with at least `MIN_USER_INTERACTIONS` applications and jobs with at least `MIN_JOB_INTERACTIONS` applications. This is applied iteratively until convergence, since removing users can cause some jobs to drop below the threshold and vice versa.

### ID remapping
After filtering, user and job IDs are remapped to a contiguous range `[0, N)`. This makes the embedding matrices compact and avoids wasting memory on inactive IDs.

| Parameter | Value | Rationale |
|---|---|---|
| `MIN_USER_INTERACTIONS` | 5 | Users with fewer interactions provide too little signal |
| `MIN_JOB_INTERACTIONS` | 5 | Unpopular jobs cannot be reliably recommended |

In [ ]:
MIN_USER_INTERACTIONS = 5
MIN_JOB_INTERACTIONS = 5

df = interactions[['user_id', 'job_id']].copy()
print(f'Before filtering: {len(df):,} interactions')

# Iterative k-core filtering
while True:
    n_before = len(df)
    user_counts = df['user_id'].value_counts()
    valid_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index
    df = df[df['user_id'].isin(valid_users)]

    job_counts = df['job_id'].value_counts()
    valid_jobs = job_counts[job_counts >= MIN_JOB_INTERACTIONS].index
    df = df[df['job_id'].isin(valid_jobs)]

    if len(df) == n_before:
        break

print(f'After filtering:  {len(df):,} interactions')
print(f'Unique users:     {df["user_id"].nunique():,}')
print(f'Unique jobs:      {df["job_id"].nunique():,}')

In [ ]:
# Remap user and job IDs to contiguous range
user_id_map = {old: new for new, old in enumerate(sorted(df['user_id'].unique()))}
job_id_map = {old: new for new, old in enumerate(sorted(df['job_id'].unique()))}

df['user_id'] = df['user_id'].map(user_id_map)
df['job_id'] = df['job_id'].map(job_id_map)

N_USERS = df['user_id'].nunique()
N_JOBS = df['job_id'].nunique()

print(f'N_USERS (remapped): {N_USERS:,}')
print(f'N_JOBS  (remapped): {N_JOBS:,}')
print(f'Density: {len(df) / (N_USERS * N_JOBS) * 100:.4f}%')

# Remap group and tier lookups
user_group = users.set_index('user_id')['group'].to_dict()
user_group_mapped = {user_id_map[old]: user_group[old] 
                     for old in user_id_map if old in user_group}

job_tier = jobs.set_index('job_id')['tier'].to_dict()
job_tier_mapped = {job_id_map[old]: job_tier[old]
                   for old in job_id_map if old in job_tier}

high_op_jobs = set(jid for jid, tier in job_tier_mapped.items() if tier == 'high_opportunity')
groupA_users = set(uid for uid, grp in user_group_mapped.items() if grp == 'A')
groupB_users = set(uid for uid, grp in user_group_mapped.items() if grp == 'B')

print(f'Group A users: {len(groupA_users):,}')
print(f'Group B users: {len(groupB_users):,}')
print(f'High-opportunity jobs: {len(high_op_jobs):,}')

---
## Step 3 - Leave-Last-Out Train / Test Split

Instead of a random split, we use a **leave-last-out** strategy: for each user, we hold out one random interaction as the test item and use all remaining interactions for training.

### Why leave-last-out?
- Every test user is **guaranteed** to have training data (unlike random splits where some users may only appear in test).
- Each user contributes exactly one test item, which makes per-user metrics more comparable.
- This is a standard evaluation protocol in implicit feedback recommender systems research.

In [ ]:
# Leave-last-out: hold out one random interaction per user
rng = np.random.default_rng(42)

test_indices = []
for uid, group in df.groupby('user_id'):
    idx = group.index.tolist()
    test_indices.append(rng.choice(idx))

test_indices = set(test_indices)

test_df = df.loc[list(test_indices)].reset_index(drop=True)
train_df = df.loc[~df.index.isin(test_indices)].reset_index(drop=True)

print(f'Train interactions: {len(train_df):,}')
print(f'Test interactions:  {len(test_df):,}')
print(f'Test users:         {test_df["user_id"].nunique():,}')

# Build lookup dictionaries
train_user_items = train_df.groupby('user_id')['job_id'].apply(set).to_dict()
test_user_items = test_df.groupby('user_id')['job_id'].apply(set).to_dict()

---
## Step 4 - Baseline Model: Implicit Matrix Factorization (BPR-SGD)

We define a latent-factor model for implicit feedback using **Bayesian Personalized Ranking (BPR)** optimised with stochastic gradient descent (SGD).

### How BPR works
For each observed positive interaction `(u, i)`, we sample a random unobserved item `j` (a job the user did *not* apply to). The model is trained to rank `i` above `j` for user `u`. The loss is:

```
L = -log sigma(x_ui - x_uj) + lambda * ||theta||^2
```

where `x_ui = P[u] . Q[i]` is the predicted score.

### Vectorised implementation

Instead of updating one pair at a time in a Python loop, we process the training data in **mini-batches** using numpy vectorised operations. This dramatically speeds up training (roughly 10-50x faster than per-sample Python loops), making it feasible to train on the full filtered dataset.

### Hyperparameters
| Parameter | Value | Description |
|---|---|---|
| `K_FACTORS` | 64 | Dimensionality of user/item latent vectors |
| `EPOCHS` | 20 | Number of full passes over the training data |
| `LR` | 0.01 | SGD learning rate |
| `REG` | 0.001 | L2 regularisation coefficient |
| `BATCH_SIZE` | 4096 | Mini-batch size for vectorised updates |

In [ ]:
class ImplicitMF:

    def __init__(self, n_users, n_items, k=64, lr=0.01, reg=0.001, seed=42):
        self.k = k
        self.lr = lr
        self.reg = reg
        self.n_items = n_items
        rng = np.random.default_rng(seed)
        scale = 0.01
        self.P = rng.normal(0, scale, (n_users, k))
        self.Q = rng.normal(0, scale, (n_items, k))

    def fit(self, df, epochs=20, batch_size=4096):
        """Train using vectorised mini-batch BPR-SGD."""
        pairs = df[['user_id', 'job_id']].values
        user_items = df.groupby('user_id')['job_id'].apply(set).to_dict()
        n = len(pairs)
        losses = []

        for epoch in range(epochs):
            # Shuffle training pairs
            idx = np.random.permutation(n)
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n, batch_size):
                batch_idx = idx[start:start + batch_size]
                batch = pairs[batch_idx]
                u = batch[:, 0]
                i = batch[:, 1]

                # Sample negative items
                j = np.random.randint(0, self.n_items, size=len(batch))
                # Resample negatives that are actually positive
                for idx_b in range(len(batch)):
                    while j[idx_b] in user_items.get(u[idx_b], set()):
                        j[idx_b] = np.random.randint(0, self.n_items)

                # Compute scores
                p_u = self.P[u]       # (batch, k)
                q_i = self.Q[i]       # (batch, k)
                q_j = self.Q[j]       # (batch, k)

                x_uij = np.sum(p_u * (q_i - q_j), axis=1)  # (batch,)

                # Sigmoid and loss
                sig = 1.0 / (1.0 + np.exp(np.clip(x_uij, -50, 50)))  # sigma(-x_uij)
                epoch_loss += -np.sum(np.log(1.0 / (1.0 + np.exp(np.clip(-x_uij, -50, 50))) + 1e-10))

                # Vectorised gradient updates
                sig_2d = sig[:, np.newaxis]  # (batch, 1)

                grad_p = sig_2d * (q_i - q_j) - self.reg * p_u
                grad_qi = sig_2d * p_u - self.reg * q_i
                grad_qj = -sig_2d * p_u - self.reg * q_j

                # Apply updates (using np.add.at for repeated indices)
                np.add.at(self.P, u, self.lr * grad_p)
                np.add.at(self.Q, i, self.lr * grad_qi)
                np.add.at(self.Q, j, self.lr * grad_qj)

                n_batches += 1

            avg_loss = epoch_loss / n
            losses.append(avg_loss)
            print(f'Epoch {epoch+1:2d}/{epochs}  loss: {avg_loss:.4f}')

        return losses

    def predict_scores(self, user):
        return self.P[user] @ self.Q.T

    def recommend(self, user, topk=10, exclude=None):
        scores = self.predict_scores(user)
        if exclude is not None:
            scores[list(exclude)] = -np.inf
        return np.argsort(scores)[::-1][:topk]

---
## Step 5 - Train the Baseline Model

We instantiate `ImplicitMF` and train it on the **full filtered training set**. Thanks to the k-core filtering (Step 2) and vectorised mini-batch updates, training on the full dataset is now computationally feasible.

The BPR loss is recorded at each epoch and plotted to verify convergence.

In [ ]:
model = ImplicitMF(N_USERS, N_JOBS, k=64, lr=0.01, reg=0.001)
losses = model.fit(train_df, epochs=20, batch_size=4096)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses)+1), losses, marker='o', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Model Training Behaviour

The BPR loss should steadily decrease across epochs, indicating that the model is learning to rank observed user-job interactions above unobserved ones. If the loss plateaus, the model has converged; if it increases, the learning rate may be too high.

---
## Step 6 - Generate Baseline Recommendations

For each test user, we generate a ranked list of the top K recommended jobs (excluding jobs already seen during training). We use K=10 as the primary evaluation cut-off, which is standard in recommendation research and produces more interpretable metrics.

In [ ]:
TOPK = 10
EVAL_USERS = min(2000, len(test_user_items))

eval_users = list(test_user_items.keys())[:EVAL_USERS]

baseline_recs = {}
for uid in tqdm(eval_users, desc='Generating recommendations'):
    exclude = train_user_items.get(uid, set())
    baseline_recs[uid] = model.recommend(uid, topk=TOPK, exclude=exclude)

print(f'Recommendations generated for {len(baseline_recs):,} users (top-{TOPK})')

---
## Step 7 - Accuracy Metrics

We evaluate the baseline recommendations using standard information-retrieval metrics.

| Metric | Description |
|---|---|
| **Hit Rate@K** | Fraction of users where the test item appears in the top-K list |
| **Precision@K** | Fraction of recommended items that are relevant |
| **Recall@K** | Fraction of relevant items that are recommended |
| **nDCG@K** | Normalised Discounted Cumulative Gain - measures ranking quality |

With leave-last-out, each user has exactly 1 test item, so Hit Rate and Recall are equivalent (a hit means the single relevant item was found).

In [ ]:
def evaluate_recs(recs_dict, test_items, k):
    """Compute accuracy metrics for a set of recommendations."""
    hits = 0
    precisions = []
    recalls = []
    ndcgs = []

    for uid, recs in recs_dict.items():
        gt = test_items.get(uid, set())
        if len(gt) == 0:
            continue

        rec_list = list(recs[:k])
        hit_list = [1 if item in gt else 0 for item in rec_list]

        n_hits = sum(hit_list)
        if n_hits > 0:
            hits += 1

        precisions.append(n_hits / k)
        recalls.append(n_hits / len(gt))

        dcg = sum(h / np.log2(r + 2) for r, h in enumerate(hit_list))
        idcg = sum(1 / np.log2(r + 2) for r in range(min(len(gt), k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    n_users = len(precisions)
    return {
        'Hit Rate@K': hits / n_users if n_users > 0 else 0,
        'Precision@K': np.mean(precisions) if precisions else 0,
        'Recall@K': np.mean(recalls) if recalls else 0,
        'nDCG@K': np.mean(ndcgs) if ndcgs else 0,
        'n_eval_users': n_users
    }

baseline_acc = evaluate_recs(baseline_recs, test_user_items, TOPK)

print(f'Baseline model metrics (K={TOPK})')
print(f'  Hit Rate@{TOPK}:   {baseline_acc["Hit Rate@K"]:.4f}')
print(f'  Precision@{TOPK}:  {baseline_acc["Precision@K"]:.4f}')
print(f'  Recall@{TOPK}:     {baseline_acc["Recall@K"]:.4f}')
print(f'  nDCG@{TOPK}:       {baseline_acc["nDCG@K"]:.4f}')
print(f'  Evaluated users: {baseline_acc["n_eval_users"]:,}')

### Interpreting the Results

With k-core filtering and leave-last-out evaluation, we expect to see meaningful metric values. A Hit Rate of, say, 0.05-0.15 means the model places the correct test item in the top-10 for 5-15% of users, which is reasonable for implicit feedback on a large item catalogue.

---
## Step 8 - Exposure Fairness Metrics

Exposure measures how often high-opportunity jobs appear in recommendation lists. Because users tend to interact more with top-ranked items, we apply a position-based discount:

```
exposure(rank) = 1 / log2(rank + 2)
```

We compute the average exposure to high-opportunity jobs separately for Group A and Group B, and report the **exposure gap** (difference between the two).

In [ ]:
def exposure_weight(rank):
    return 1.0 / np.log2(rank + 2)

def compute_exposure(recs_dict, high_op_set, group_a_set, group_b_set):
    """Compute per-group average exposure to high-opportunity jobs."""
    exp_a = []
    exp_b = []

    for uid, recs in recs_dict.items():
        user_exp = sum(exposure_weight(r) for r, job in enumerate(recs) if job in high_op_set)
        if uid in group_a_set:
            exp_a.append(user_exp)
        elif uid in group_b_set:
            exp_b.append(user_exp)

    mean_a = np.mean(exp_a) if exp_a else 0
    mean_b = np.mean(exp_b) if exp_b else 0
    return mean_a, mean_b

exp_A_mean, exp_B_mean = compute_exposure(baseline_recs, high_op_jobs, groupA_users, groupB_users)

print(f'Exposure Group A: {exp_A_mean:.4f}')
print(f'Exposure Group B: {exp_B_mean:.4f}')
print(f'Exposure gap (A - B): {exp_A_mean - exp_B_mean:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Group A', 'Group B'], [exp_A_mean, exp_B_mean], color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Average Exposure to High-Opportunity Jobs')
ax.set_title('Baseline: Exposure by Group')
ax.bar_label(bars, fmt='%.4f', padding=3)
plt.tight_layout()
plt.show()

### Exposure Fairness Analysis

The exposure gap indicates whether one group systematically receives more visibility of high-opportunity jobs. A positive gap (A > B) suggests that experienced users (Group A) are favoured; a negative gap suggests the opposite.

---
## Step 9 - Fairness-Aware Re-ranking

A post-processing algorithm re-orders each user's candidate list to increase fairness without fully discarding relevance. The underlying model is unchanged; only the final ranking is adjusted.

### Algorithm
We first generate a larger candidate set (top-100 from the baseline model) and then re-score using:

```
score(item) = relevance_score - beta * exposure_penalty(rank)
```

where `exposure_penalty = 1 / log2(rank + 2)` and `beta` controls fairness strength. The top-K items from the re-scored list become the final recommendations.

### Parameters
| Parameter | Default | Effect |
|---|---|---|
| `BETA` | 0.5 | Fairness strength. Higher = more fairness, less accuracy |
| `CANDIDATE_SIZE` | 100 | Number of candidates to re-rank per user |

In [ ]:
BETA = 0.5
CANDIDATE_SIZE = 100

def generate_fair_recs(model, eval_users, train_user_items, beta, candidate_size, topk):
    """Generate fairness-aware recommendations via re-ranking."""
    fair_recs = {}
    for uid in eval_users:
        exclude = train_user_items.get(uid, set())
        # Get larger candidate set with raw scores
        scores = model.predict_scores(uid)
        scores[list(exclude)] = -np.inf
        top_candidates = np.argsort(scores)[::-1][:candidate_size]

        # Re-score with fairness penalty
        adjusted = []
        for rank, job in enumerate(top_candidates):
            relevance = scores[job]
            penalty = exposure_weight(rank)
            adjusted.append((job, relevance - beta * penalty))

        adjusted.sort(key=lambda x: x[1], reverse=True)
        fair_recs[uid] = np.array([j for j, _ in adjusted[:topk]])

    return fair_recs

fair_recs = generate_fair_recs(model, eval_users, train_user_items, BETA, CANDIDATE_SIZE, TOPK)
print(f'Fair recommendations generated: {len(fair_recs):,}')

---
## Step 10 - Evaluate Fair Model

We compute the same accuracy and exposure metrics for the fairness-aware recommendations.

In [ ]:
fair_acc = evaluate_recs(fair_recs, test_user_items, TOPK)
fair_exp_A, fair_exp_B = compute_exposure(fair_recs, high_op_jobs, groupA_users, groupB_users)

print(f'Fair model metrics (K={TOPK}, beta={BETA})')
print(f'  Hit Rate@{TOPK}:   {fair_acc["Hit Rate@K"]:.4f}')
print(f'  Precision@{TOPK}:  {fair_acc["Precision@K"]:.4f}')
print(f'  Recall@{TOPK}:     {fair_acc["Recall@K"]:.4f}')
print(f'  nDCG@{TOPK}:       {fair_acc["nDCG@K"]:.4f}')
print(f'  Exposure A:     {fair_exp_A:.4f}')
print(f'  Exposure B:     {fair_exp_B:.4f}')
print(f'  Exposure gap:   {fair_exp_A - fair_exp_B:.4f}')

---
## Step 11 - Compare: Baseline vs. Fair

A side-by-side comparison of the baseline and fairness-aware models on all metrics.

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Baseline', 'Fair'],
    'Hit Rate@K': [baseline_acc['Hit Rate@K'], fair_acc['Hit Rate@K']],
    'Precision@K': [baseline_acc['Precision@K'], fair_acc['Precision@K']],
    'Recall@K': [baseline_acc['Recall@K'], fair_acc['Recall@K']],
    'nDCG@K': [baseline_acc['nDCG@K'], fair_acc['nDCG@K']],
    'Exposure A': [exp_A_mean, fair_exp_A],
    'Exposure B': [exp_B_mean, fair_exp_B],
    'Exposure Gap': [exp_A_mean - exp_B_mean, fair_exp_A - fair_exp_B]
})

comparison

---
## Step 12 - Trade-off Frontier: Sweeping Beta

We run the fairness re-ranker for several values of beta, recording both accuracy and fairness at each point. The resulting frontier shows stakeholders the cost (in accuracy) of reducing the exposure gap.

In [ ]:
betas = [0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
sweep_results = []

for beta in tqdm(betas, desc='Sweeping beta'):
    recs = generate_fair_recs(model, eval_users, train_user_items, beta, CANDIDATE_SIZE, TOPK)
    acc = evaluate_recs(recs, test_user_items, TOPK)
    e_a, e_b = compute_exposure(recs, high_op_jobs, groupA_users, groupB_users)
    sweep_results.append({
        'beta': beta,
        'Hit Rate@K': acc['Hit Rate@K'],
        'nDCG@K': acc['nDCG@K'],
        'Recall@K': acc['Recall@K'],
        'Exposure Gap': e_a - e_b,
        'Exp A': e_a,
        'Exp B': e_b
    })

sweep_df = pd.DataFrame(sweep_results)
sweep_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Hit Rate vs beta
axes[0].plot(sweep_df['beta'], sweep_df['Hit Rate@K'], marker='o', color='#4C72B0')
axes[0].set_xlabel('Beta (fairness strength)')
axes[0].set_ylabel('Hit Rate@K')
axes[0].set_title('Accuracy vs Fairness Strength')
axes[0].grid(True, alpha=0.3)

# Plot 2: Exposure gap vs beta
axes[1].plot(sweep_df['beta'], sweep_df['Exposure Gap'].abs(), marker='s', color='#DD8452')
axes[1].set_xlabel('Beta (fairness strength)')
axes[1].set_ylabel('|Exposure Gap|')
axes[1].set_title('Exposure Gap vs Fairness Strength')
axes[1].grid(True, alpha=0.3)

# Plot 3: Hit Rate vs Exposure Gap (Pareto frontier)
axes[2].plot(sweep_df['Exposure Gap'].abs(), sweep_df['Hit Rate@K'], marker='D', color='#55A868')
for _, row in sweep_df.iterrows():
    axes[2].annotate(f'b={row["beta"]}', (abs(row['Exposure Gap']), row['Hit Rate@K']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[2].set_xlabel('|Exposure Gap|')
axes[2].set_ylabel('Hit Rate@K')
axes[2].set_title('Accuracy-Fairness Trade-off')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Trade-off Interpretation

The three plots above show:

1. **Accuracy vs beta**: How Hit Rate decreases as fairness strength increases.
2. **Exposure gap vs beta**: How the exposure disparity between groups shrinks with higher beta.
3. **Pareto frontier**: The direct trade-off between accuracy and fairness. Stakeholders can use this curve to choose a beta value that balances both objectives.

---
## Step 13 - Conclusion

This notebook implemented a fairness-aware job recommender system using implicit matrix factorization with Bayesian Personalized Ranking (BPR).

**Key findings:**

- **Data preprocessing matters**: K-core filtering reduced the interaction matrix to active users and popular jobs, dramatically improving the quality of learned embeddings and producing meaningful evaluation metrics.
- **Leave-last-out evaluation**: This protocol ensures every test user has training data and provides a cleaner signal than random splits.
- **Vectorised BPR**: Mini-batch training with numpy vectorisation made it feasible to train on the full filtered dataset within reasonable time.
- **Exposure analysis**: The baseline recommender may show a measurable exposure gap between Group A (managed others) and Group B (did not manage others) in terms of access to high-opportunity jobs.
- **Fairness-accuracy trade-off**: The beta sweep demonstrates that reducing the exposure gap comes at a cost to recommendation accuracy. The Pareto frontier helps stakeholders make informed decisions about this balance.

**Limitations and future work:**

- The leave-last-out protocol uses a random held-out item rather than a temporal split. A time-based split would better simulate real deployment.
- The re-ranking approach only adjusts the final ranking. An in-processing approach (e.g. fairness-aware loss function) could produce better results.
- Group definitions are binary and based on a single feature. Real-world fairness studies would consider intersectional groups and multiple protected attributes.